# Ridge regression: IMU embeddings to walking measures

This notebook tests whether each embedding block can predict full-signal Xsens sacrum walking measures. The four embedding blocks (generic and stroke-adapted LIMU-BERT-X and HART) are evaluated separately using their full embedding dimensions, not PCA scores.

The analysis averages NW trial embeddings and gait measures within each participant, producing one row per participant. It uses nested cross-validation: the outer leave-one-participant-out loop evaluates predictions on held-out participants, while an inner four-fold loop selects the ridge regularization strength. Feature scaling and model tuning occur within the training folds. With 17 participants, results are exploratory.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, LeaveOneOut, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, pearsonr

search_paths=[Path.cwd(),*Path.cwd().parents]
repo_root=next((p for p in search_paths if (p/'Data/analysis/master_analysis_table_pca.csv').exists()),None)
if repo_root is None:
    raise FileNotFoundError('Could not find Data/analysis/master_analysis_table_pca.csv from the current directory.')
master=pd.read_csv(repo_root/'Data/analysis/master_analysis_table_pca.csv')
results_dir=repo_root/'Data/analysis/ridge_results'
results_dir.mkdir(parents=True,exist_ok=True)
print(f'Loaded {len(master)} trial rows from {master.participant_id.nunique()} participants.')

In [ ]:
outcomes={
    'Step count':'sacrum_walk_step_count',
    'Cadence (steps/min)':'sacrum_walk_cadence_steps_min',
    'Mean step time (s)':'sacrum_walk_mean_step_time_s',
    'Walking duration (s)':'sacrum_walk_duration_s',
}
embedding_blocks={
    'LIMU-BERT-X generic':[c for c in master if c.startswith('limu_generic_embedding_')],
    'LIMU-BERT-X stroke-adapted':[c for c in master if c.startswith('limu_stroke_adapted_embedding_')],
    'HART generic':[c for c in master if c.startswith('hart_generic_embedding_')],
    'HART stroke-adapted':[c for c in master if c.startswith('hart_stroke_adapted_embedding_')],
}
participant_targets=master.groupby('participant_id')[list(outcomes.values())].mean()
participant_embeddings={
    name:master.groupby('participant_id')[columns].mean()
    for name,columns in embedding_blocks.items()
}
print('Participants:',len(participant_targets))
for name,columns in embedding_blocks.items():
    print(f'{name}: {len(columns)} embedding dimensions')

## Nested participant-level cross-validation

For each outer fold, one participant is held out for testing. The other participants are used to choose the ridge penalty in an inner four-fold cross-validation. The standardizer is part of the pipeline, so it is fitted only on training data in each fold. A training-mean prediction is included as a simple baseline.

In [ ]:
alphas=np.logspace(-4,6,21)
outer_cv=LeaveOneOut()
inner_cv=KFold(n_splits=4,shuffle=True,random_state=42)
all_predictions=[]
summary_rows=[]

for embedding_name,features in participant_embeddings.items():
    for outcome_name,target_col in outcomes.items():
        joined=features.join(participant_targets[[target_col]],how='inner').dropna()
        X=joined[features.columns].to_numpy(dtype=float)
        y=joined[target_col].to_numpy(dtype=float)
        participants=joined.index.to_numpy()
        predictions=np.empty(len(y),dtype=float)
        baseline_predictions=np.empty(len(y),dtype=float)
        selected_alphas=[]
        for train_idx,test_idx in outer_cv.split(X,y):
            pipeline=Pipeline([('scale',StandardScaler()),('ridge',Ridge())])
            search=GridSearchCV(
                pipeline,
                param_grid={'ridge__alpha':alphas},
                scoring='neg_mean_absolute_error',
                cv=inner_cv,
                n_jobs=1,
            )
            search.fit(X[train_idx],y[train_idx])
            predictions[test_idx]=search.best_estimator_.predict(X[test_idx])
            baseline_predictions[test_idx]=y[train_idx].mean()
            selected_alphas.append(search.best_params_['ridge__alpha'])
            all_predictions.append({
                'embedding_block':embedding_name,
                'outcome':outcome_name,
                'participant_id':participants[test_idx[0]],
                'observed':y[test_idx[0]],
                'predicted':predictions[test_idx[0]],
                'baseline_predicted':baseline_predictions[test_idx[0]],
                'selected_alpha':search.best_params_['ridge__alpha'],
            })
        summary_rows.append({
            'embedding_block':embedding_name,
            'outcome':outcome_name,
            'n_participants':len(y),
            'embedding_dimensions':X.shape[1],
            'LOPO_MAE':mean_absolute_error(y,predictions),
            'LOPO_RMSE':mean_squared_error(y,predictions)**0.5,
            'LOPO_R2':r2_score(y,predictions),
            'baseline_mean_MAE':mean_absolute_error(y,baseline_predictions),
            'median_selected_alpha':float(np.median(selected_alphas)),
            'fraction_folds_at_max_alpha':float(np.mean(np.isclose(selected_alphas,alphas[-1]))),
        })

summary=pd.DataFrame(summary_rows)
prediction_table=pd.DataFrame(all_predictions)
summary.to_csv(results_dir/'ridge_lopo_summary.csv',index=False)
prediction_table.to_csv(results_dir/'ridge_lopo_predictions.csv',index=False)
display(summary.round(3))

## Paired generic-versus-adapted error comparison

For each outcome, compare generic and stroke-adapted absolute errors for the same held-out participant. The difference is defined as generic error minus adapted error: a positive value favors the adapted model. The percentile bootstrap interval is descriptive with this small sample; it is not a confirmatory test.

In [ ]:
paired_predictions=prediction_table.copy()
paired_predictions['absolute_error']=(paired_predictions['observed']-paired_predictions['predicted']).abs()
family_pairs={
    'LIMU-BERT-X':('LIMU-BERT-X generic','LIMU-BERT-X stroke-adapted'),
    'HART':('HART generic','HART stroke-adapted'),
}
rng=np.random.default_rng(2026)
paired_rows=[]
for family,(generic_name,adapted_name) in family_pairs.items():
    for outcome_name in outcomes:
        subset=paired_predictions[paired_predictions.outcome==outcome_name]
        errors=subset.pivot(index='participant_id',columns='embedding_block',values='absolute_error')
        errors=errors[[generic_name,adapted_name]].dropna()
        delta=(errors[generic_name]-errors[adapted_name]).to_numpy()
        draws=delta[rng.integers(0,len(delta),size=(20000,len(delta)))].mean(axis=1)
        ci_low,ci_high=np.quantile(draws,[.025,.975])
        paired_rows.append({
            'model_family':family,
            'outcome':outcome_name,
            'n_participants':len(delta),
            'generic_MAE':errors[generic_name].mean(),
            'adapted_MAE':errors[adapted_name].mean(),
            'mean_delta_generic_minus_adapted':delta.mean(),
            'median_delta_generic_minus_adapted':np.median(delta),
            'adapted_lower_error_n':int((delta < 0).sum()),
            'generic_lower_error_n':int((delta > 0).sum()),
            'bootstrap_95pct_low':ci_low,
            'bootstrap_95pct_high':ci_high,
        })
paired_summary=pd.DataFrame(paired_rows)
paired_summary.to_csv(results_dir/'ridge_paired_comparison_summary.csv',index=False)
display(paired_summary.round(3))

## Observed versus predicted values

Each point is one held-out participant. Points closer to the dashed line have more accurate predictions. These plots summarize leave-one-participant-out predictions; they are not fitted-value plots.

In [ ]:
def plot_outcome(outcome_name):
    pred=prediction_table[prediction_table.outcome==outcome_name]
    fig,ax=plt.subplots(figsize=(6,6))
    for name,group in pred.groupby('embedding_block',sort=False):
        ax.scatter(group.observed,group.predicted,label=name,alpha=.8,s=42)
    low=min(pred.observed.min(),pred.predicted.min())
    high=max(pred.observed.max(),pred.predicted.max())
    ax.plot([low,high],[low,high],'k--',linewidth=1)
    ax.set(xlabel=f'Observed {outcome_name}',ylabel=f'LOPO predicted {outcome_name}',title=f'Ridge predictions: {outcome_name}')
    ax.legend(fontsize=8)
    ax.grid(alpha=.2)
    fig.tight_layout()
    slug=outcome_name.lower().replace(' ','_').replace('/','_').replace('(','').replace(')','')
    fig.savefig(results_dir/f'{slug}_observed_vs_predicted.png',dpi=160)
    plt.show()

### Step count

In [ ]:
plot_outcome('Step count')

### Cadence (steps/min)

In [ ]:
plot_outcome('Cadence (steps/min)')

### Mean step time (s)

In [ ]:
plot_outcome('Mean step time (s)')

### Walking duration (s)

In [ ]:
plot_outcome('Walking duration (s)')

## Interpretation cautions

- The outer test fold contains a participant not used in that fold's model fitting or hyperparameter selection.
- Lower MAE and RMSE are better. LOPO R² below zero means the predictions are worse than the overall-mean reference used by the R² formula. `baseline_mean_MAE` compares against a feasible prediction: the mean gait measure from the training participants in each fold.
- The alpha grid is deliberately broad. `fraction_folds_at_max_alpha` flags whether tuning still selects its upper boundary.
- Only 17 participants are available. Performance estimates and selected ridge penalties may be unstable and should be described as exploratory.
- Step count, cadence, step time, and duration are related outcomes, not four independent confirmations. Fall-risk labels are not model inputs or targets in this regression analysis.
- This analysis asks whether the embeddings predict these extracted lab measures. It does not establish clinical validity or stroke specificity.